# Sistema de Calidad de Datos Bancarios con IA Multi-Agente
## Evaluación experimental — Notebook v4 (Mejorado)

Este notebook implementa un sistema completo de calidad de datos bancarios donde:

1. **Multi-Agente LLM**: Pipeline de 4 agentes especializados (Intérprete → Codificador → Validador → Corrector)
2. **RAG (Retrieval Augmented Generation)**: Base de conocimiento con documentación de pandas, ejemplos exitosos y schema del dataset
3. **Validación AST estricta**: Verificación de seguridad y calidad del código generado antes de ejecutar
4. **Fine-tuning con 30+ ejemplos** incluyendo chain-of-thought y manejo explícito de nulos
5. **Modelos de ML**: Autoencoder + Isolation Forest para anomalías, Random Forest para predicción

**Diferencial vs versión anterior**: El LLM pasó de 80% (4/5 reglas) a un enfoque robusto con validación y corrección automática.

## 1. Configuración del entorno

In [1]:
import json
import re
import ast
import subprocess
import time
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

OLLAMA_MODEL = "calidad-datos-v4"  # Modelo fine-tuneado
REFERENCE_DATE = pd.Timestamp("2026-06-01")
print(f"Modelo: {OLLAMA_MODEL} | Referencia: {REFERENCE_DATE.date()}")

Modelo: calidad-datos-v4 | Referencia: 2026-06-01


## 2. Motor de inferencia LLM + Base de conocimiento RAG

El sistema RAG inyecta contexto relevante en cada prompt:
- Schema del DataFrame con tipos y nulos posibles
- Documentación de operaciones pandas vectorizadas seguras
- Ejemplos exitosos anteriores
- Reglas de negocio del banco

In [2]:
import requests

def call_ollama(prompt, model=None, timeout=180):
    """Invoca Ollama via HTTP API."""
    selected = model or OLLAMA_MODEL
    try:
        resp = requests.post(
            "http://127.0.0.1:11434/api/generate",
            json={"model": selected, "prompt": prompt, "stream": False,
                  "options": {"temperature": 0.1, "num_predict": 600}},
            timeout=timeout
        )
        return resp.json().get("response", "").strip()
    except Exception as e:
        return f"ERROR_OLLAMA: {e}"

# Verificar conexion
test = call_ollama("Responde solo OK")
print(f"Ollama: {'Conectado' if 'ERROR' not in test else test}")

Ollama: Conectado


In [3]:
# ============================================================
# BASE DE CONOCIMIENTO RAG
# ============================================================

# Schema del dataset (inyectado en cada prompt)
DATASET_SCHEMA = """
SCHEMA DEL DATAFRAME (df):
| Columna       | Tipo     | Nulos posibles | Descripcion                    |
|---------------|----------|----------------|--------------------------------|
| customer_id   | str      | No             | ID unico del cliente           |
| document_id   | str/NaN  | Si             | Documento 8 digitos            |
| segment       | str/NaN  | Si             | Mass,Standard,Premium,Corporate|
| product       | str/NaN  | Si             | Savings,Checking,TermDeposit,CreditCard |
| balance       | float/NaN| Si             | Saldo (>=0 para depositos)     |
| last_txn_date | str/NaT  | Si             | Fecha YYYY-MM-DD <= 2026-06-01 |
| phone         | str/NaN  | Si             | Exactamente 9 digitos          |
| email         | str/NaN  | Si             | Formato user@domain.ext        |
"""

# Reglas de codigo seguro (prohibiciones)
CODE_RULES = """
REGLAS OBLIGATORIAS DE GENERACION DE CODIGO:
1. NUNCA uses .apply(lambda ...) — solo operaciones vectorizadas de pandas.
2. SIEMPRE usa .fillna('').astype(str) ANTES de cualquier operacion .str
3. SIEMPRE usa pd.to_numeric(col, errors='coerce') antes de comparaciones numericas.
4. SIEMPRE usa pd.to_datetime(col, errors='coerce') antes de comparaciones de fecha.
5. La funcion recibe UN parametro (df) y retorna el DataFrame limpio.
6. Usa df.copy() al inicio. NO modifiques el original.
7. NO uses eval(), exec(), __import__, os, subprocess.
8. Formato de respuesta: SOLO un bloque ```python``` con la funcion completa.
"""

# Documentacion de pandas segura
PANDAS_SAFE_OPS = """
OPERACIONES PANDAS SEGURAS (usa estas):
- df["col"].fillna("").astype(str).str.fullmatch(pattern)
- df["col"].fillna("").astype(str).str.replace(r"[^0-9]", "", regex=True)
- df["col"].fillna("").astype(str).str.slice(0, N).str.zfill(N)
- pd.to_numeric(df["col"], errors="coerce")
- pd.to_datetime(df["col"], errors="coerce")
- df["col"].isin(valid_set)
- df.duplicated(subset=["col"], keep="first")
- cleaned.loc[mask, "col"] = valor_corregido
"""

# Ejemplos exitosos (se acumulan durante ejecucion)
SUCCESSFUL_EXAMPLES = []

def build_rag_context(rule_text, extra_context=""):
    """Construye el contexto RAG completo para un prompt."""
    examples_section = ""
    if SUCCESSFUL_EXAMPLES:
        examples_section = "\nEJEMPLOS EXITOSOS ANTERIORES:\n"
        for ex in SUCCESSFUL_EXAMPLES[-3:]:  # Ultimos 3
            examples_section += f"Regla: {ex['rule']}\nCodigo:\n{ex['code']}\n---\n"
    
    return f"""{DATASET_SCHEMA}
{CODE_RULES}
{PANDAS_SAFE_OPS}
{examples_section}
{extra_context}"""

print("Base de conocimiento RAG cargada.")
print(f"  Schema: {len(DATASET_SCHEMA)} chars")
print(f"  Reglas de codigo: {len(CODE_RULES)} chars")
print(f"  Pandas docs: {len(PANDAS_SAFE_OPS)} chars")

Base de conocimiento RAG cargada.
  Schema: 827 chars
  Reglas de codigo: 623 chars
  Pandas docs: 445 chars


## 3. Validación AST estricta

Antes de ejecutar cualquier código generado por el LLM, se valida:
- Sintaxis correcta (ast.parse)
- Sin imports peligrosos (os, subprocess, eval, exec)
- Función con exactamente 1 parámetro `df`
- Termina con `return`
- Sin `apply(lambda` (fuerza vectorización)

In [4]:
# ============================================================
# VALIDADOR AST DE CODIGO GENERADO
# ============================================================

@dataclass
class ValidationResult:
    is_valid: bool
    errors: List[str] = field(default_factory=list)
    function_name: Optional[str] = None

FORBIDDEN_NAMES = {"eval", "exec", "__import__", "os", "subprocess", "system", "open"}
FORBIDDEN_PATTERNS = ["apply(lambda", ".apply(lambda", "exec(", "eval(", "__import__"]

def validate_generated_code(code_str: str, expected_func_name: str = None) -> ValidationResult:
    """Valida codigo generado por el LLM usando AST."""
    errors = []
    
    # 1. Verificar sintaxis
    try:
        tree = ast.parse(code_str)
    except SyntaxError as e:
        return ValidationResult(False, [f"SyntaxError: {e}"])
    
    # 2. Buscar la funcion principal
    func_defs = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
    if not func_defs:
        errors.append("No se encontro ninguna definicion de funcion (def ...)")
        return ValidationResult(False, errors)
    
    main_func = func_defs[0]
    func_name = main_func.name
    
    # 3. Verificar parametro unico 'df'
    args = main_func.args
    if len(args.args) != 1:
        errors.append(f"La funcion debe tener exactamente 1 parametro (df), tiene {len(args.args)}")
    elif args.args[0].arg != 'df':
        errors.append(f"El parametro debe llamarse 'df', se llama '{args.args[0].arg}'")
    
    # 4. Verificar que termina con return
    last_stmt = main_func.body[-1] if main_func.body else None
    if not isinstance(last_stmt, ast.Return):
        errors.append("La funcion debe terminar con 'return cleaned' o 'return df'")
    
    # 5. Buscar imports/calls prohibidos
    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            errors.append(f"Uso prohibido: '{node.id}'")
        if isinstance(node, ast.Import) or isinstance(node, ast.ImportFrom):
            module = ""
            if isinstance(node, ast.Import):
                module = node.names[0].name
            elif node.module:
                module = node.module
            if module in {"os", "subprocess", "sys", "shutil"}:
                errors.append(f"Import prohibido: '{module}'")
    
    # 6. Verificar patrones prohibidos en texto
    for pattern in FORBIDDEN_PATTERNS:
        if pattern in code_str:
            errors.append(f"Patron prohibido detectado: '{pattern}'")
    
    is_valid = len(errors) == 0
    return ValidationResult(is_valid, errors, func_name)

# Test del validador
test_code = '''def clean_test(df):
    cleaned = df.copy()
    cleaned.loc[cleaned["balance"] < 0, "balance"] = 0.0
    return cleaned'''

result = validate_generated_code(test_code)
print(f"Test validador: {'PASS' if result.is_valid else 'FAIL'} - {result.errors}")

test_bad = '''def bad(df):
    return df.apply(lambda x: x.strip())'''
result2 = validate_generated_code(test_bad)
print(f"Test codigo malo: {'PASS' if result2.is_valid else 'FAIL'} - {result2.errors}")

Test validador: PASS - []
Test codigo malo: FAIL - ["Patron prohibido detectado: 'apply(lambda'", "Patron prohibido detectado: '.apply(lambda'"]


## 4. Pipeline Multi-Agente

El sistema divide el procesamiento de cada regla en 4 agentes especializados:

| Agente | Rol | Input | Output |
|--------|-----|-------|--------|
| **Intérprete** | Identifica dimensión, columnas, condición | Regla NL | JSON estructurado |
| **Codificador** | Genera código Python vectorizado | JSON + RAG | Código Python |
| **Validador** | Verifica seguridad y corrección AST | Código | Pass/Fail + errores |
| **Corrector** | Diagnostica y corrige errores | Código + Error + Datos | Código corregido |

In [5]:
# ============================================================
# AGENTE 1: INTERPRETE
# ============================================================

def agent_interpreter(rule_text: str) -> Dict:
    """Interpreta una regla NL y extrae estructura."""
    prompt = f"""Analiza esta regla de calidad de datos bancarios y responde SOLO con JSON valido.

REGLA: "{rule_text}"

{DATASET_SCHEMA}

Responde EXACTAMENTE con este formato JSON (sin texto adicional):
{{
  "dimension": "Completitud|Exactitud|Consistencia|Unicidad|Validez",
  "columns_affected": ["col1", "col2"],
  "condition_type": "null_check|range_check|format_check|duplicate_check|cross_field_check",
  "has_nulls_risk": true,
  "correction_action": "descripcion breve de la correccion"
}}"""
    
    response = call_ollama(prompt)
    
    # Extraer JSON de la respuesta
    try:
        # Buscar JSON en la respuesta
        json_match = re.search(r'\{[^{}]+\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass
    
    # Fallback: interpretar manualmente
    return {
        "dimension": "Validez",
        "columns_affected": ["unknown"],
        "condition_type": "format_check",
        "has_nulls_risk": True,
        "correction_action": rule_text
    }

# Test
test_interp = agent_interpreter("El saldo no debe ser negativo para productos de deposito.")
print("Agente Interprete - Test:")
print(json.dumps(test_interp, indent=2, ensure_ascii=False))

Agente Interprete - Test:
{
  "dimension": "Exactitud",
  "columns_affected": [
    "balance"
  ],
  "condition_type": "range_check",
  "has_nulls_risk": false,
  "correction_action": "poner balance = 0.0 si es < 0 para productos de depósito"
}


In [6]:
# ============================================================
# AGENTE 2: CODIFICADOR
# ============================================================

def agent_coder(rule_text: str, interpretation: Dict, func_name: str) -> str:
    """Genera codigo Python basado en la interpretacion estructurada."""
    
    # Construir el template segun el tipo de condicion
    null_handling = ""
    if interpretation.get("has_nulls_risk", True):
        cols = interpretation.get("columns_affected", [])
        for col in cols:
            null_handling += f"    # Manejo de nulos en '{col}'\n"
    
    rag_context = build_rag_context(rule_text)
    
    prompt = f"""Genera UNA funcion Python que implemente esta regla de calidad de datos.

REGLA: "{rule_text}"
INTERPRETACION: {json.dumps(interpretation, ensure_ascii=False)}
NOMBRE DE FUNCION: {func_name}

{rag_context}

TEMPLATE OBLIGATORIO (completa solo la seccion marcada):
```python
def {func_name}(df):
    cleaned = df.copy()
    # === TU CODIGO AQUI (solo operaciones vectorizadas) ===
    # RECUERDA: .fillna('').astype(str) antes de .str operations
    # RECUERDA: pd.to_numeric(..., errors='coerce') antes de comparaciones
    # === FIN DE TU CODIGO ===
    return cleaned
# Dimension: {interpretation.get('dimension', 'Validez')}
```

RESPONDE SOLO CON EL BLOQUE ```python```. Nada mas."""
    
    return call_ollama(prompt)


# ============================================================
# AGENTE 3: VALIDADOR (usa AST)
# ============================================================

def agent_validator(code_str: str, func_name: str, df_sample: pd.DataFrame) -> Tuple[bool, str]:
    """Valida codigo con AST + ejecucion en sandbox."""
    
    # Paso 1: Validacion AST
    ast_result = validate_generated_code(code_str, func_name)
    if not ast_result.is_valid:
        return False, f"AST errors: {ast_result.errors}"
    
    # Paso 2: Ejecucion en sandbox con muestra pequena
    safe_ns = {"pd": pd, "np": np, "re": re}
    try:
        exec(code_str, safe_ns)
        func = safe_ns.get(func_name)
        if func is None:
            return False, f"Funcion '{func_name}' no encontrada tras exec"
        
        # Ejecutar sobre muestra
        result_df = func(df_sample.head(10).copy())
        
        # Verificar resultado
        if not isinstance(result_df, pd.DataFrame):
            return False, "La funcion no retorna un DataFrame"
        if list(result_df.columns) != list(df_sample.columns):
            return False, f"Columnas cambiaron: {list(result_df.columns)}"
        
        return True, "OK"
    except Exception as e:
        return False, f"Runtime error: {type(e).__name__}: {e}"


# ============================================================
# AGENTE 4: CORRECTOR
# ============================================================

def agent_corrector(original_code: str, error_msg: str, rule_text: str,
                    interpretation: Dict, func_name: str, df_sample: pd.DataFrame) -> str:
    """Diagnostica y corrige codigo que fallo."""
    
    # Muestra de datos problematicos
    sample_str = df_sample.head(3).to_string()
    
    prompt = f"""El siguiente codigo Python FALLO con este error. Corrigelo.

CODIGO ORIGINAL:
```python
{original_code}
```

ERROR: {error_msg}

MUESTRA DE DATOS (3 filas):
{sample_str}

REGLA ORIGINAL: "{rule_text}"

{CODE_RULES}

ERRORES COMUNES Y SOLUCIONES:
- "expected string or bytes-like object, got 'float'" → Falta .fillna('').astype(str) antes de .str
- "Cannot perform 'rand_' with a dtyped [bool]" → Usar .fillna(False) en mascaras booleanas
- "invalid type comparison" → Usar pd.to_numeric(col, errors='coerce') antes de < o >

Genera SOLO el codigo corregido en un bloque ```python```. La funcion debe llamarse '{func_name}'."""
    
    return call_ollama(prompt)

print("Pipeline Multi-Agente configurado:")
print("  1. Interprete: regla NL -> JSON estructurado")
print("  2. Codificador: JSON + RAG -> codigo Python")
print("  3. Validador: AST + sandbox -> pass/fail")
print("  4. Corrector: error + datos -> codigo corregido")

Pipeline Multi-Agente configurado:
  1. Interprete: regla NL -> JSON estructurado
  2. Codificador: JSON + RAG -> codigo Python
  3. Validador: AST + sandbox -> pass/fail
  4. Corrector: error + datos -> codigo corregido


## 5. Orquestador del pipeline

Coordina los 4 agentes para cada regla con reintentos inteligentes.

In [7]:
# ============================================================
# ORQUESTADOR MULTI-AGENTE
# ============================================================

@dataclass
class RuleResult:
    rule_text: str
    dimension: str
    func_name: str
    code: str
    success: bool
    attempts: int
    errors_history: List[str] = field(default_factory=list)
    before_issues: int = 0
    after_issues: int = 0

def extract_code_block(response: str) -> str:
    """Extrae bloque ```python``` de la respuesta."""
    pattern = r'```(?:python)?\s*\n(.*?)```'
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()
    # Fallback: buscar def ...
    lines = response.split('\n')
    code_lines = []
    in_func = False
    for line in lines:
        if line.strip().startswith('def '):
            in_func = True
        if in_func:
            code_lines.append(line)
            if line.strip().startswith('return') and in_func:
                break
    return '\n'.join(code_lines) if code_lines else response

def process_rule(rule_text: str, func_name: str, df: pd.DataFrame, max_attempts: int = 4) -> RuleResult:
    """Procesa una regla a traves del pipeline multi-agente completo."""
    
    print(f"  [Interprete] Analizando regla...")
    interpretation = agent_interpreter(rule_text)
    dimension = interpretation.get("dimension", "Validez")
    
    errors_history = []
    code_str = ""
    
    for attempt in range(1, max_attempts + 1):
        print(f"  [Codificador] Intento {attempt}/{max_attempts}...")
        
        if attempt == 1:
            # Primera vez: generar desde cero
            raw_response = agent_coder(rule_text, interpretation, func_name)
        else:
            # Reintentos: usar corrector
            print(f"  [Corrector] Corrigiendo error: {errors_history[-1][:60]}...")
            raw_response = agent_corrector(
                code_str, errors_history[-1], rule_text, interpretation, func_name, df
            )
        
        code_str = extract_code_block(raw_response)
        
        # Validar con AST + sandbox
        print(f"  [Validador] Verificando codigo...")
        is_valid, error_msg = agent_validator(code_str, func_name, df)
        
        if is_valid:
            print(f"  [OK] Codigo valido en intento {attempt}")
            # Registrar como ejemplo exitoso para RAG
            SUCCESSFUL_EXAMPLES.append({"rule": rule_text, "code": code_str})
            return RuleResult(
                rule_text=rule_text, dimension=dimension,
                func_name=func_name, code=code_str,
                success=True, attempts=attempt, errors_history=errors_history
            )
        else:
            errors_history.append(error_msg)
            print(f"  [FAIL] {error_msg[:80]}")
    
    # Todos los intentos fallaron
    return RuleResult(
        rule_text=rule_text, dimension=dimension,
        func_name=func_name, code=code_str,
        success=False, attempts=max_attempts, errors_history=errors_history
    )

print("Orquestador listo.")

Orquestador listo.


## 6. Datos sintéticos bancarios

In [8]:
np.random.seed(42)
n = 250
rng = np.random.default_rng(42)

segments = ['Mass', 'Standard', 'Premium', 'Corporate']
products = ['Savings', 'Checking', 'TermDeposit', 'CreditCard']

seg_choices = rng.choice(segments, n, p=[0.35, 0.35, 0.20, 0.10])
prod_choices = rng.choice(products, n, p=[0.30, 0.25, 0.25, 0.20])

base_balance = np.where(seg_choices == 'Premium', rng.normal(8000, 2000, n),
               np.where(seg_choices == 'Corporate', rng.normal(15000, 4000, n),
               rng.normal(2000, 800, n)))

df_raw = pd.DataFrame({
    'customer_id': [f'CUST{i:05d}' for i in range(1, n+1)],
    'document_id': [f'{rng.integers(10000000, 99999999):08d}' for _ in range(n)],
    'segment': seg_choices,
    'product': prod_choices,
    'balance': np.round(base_balance, 2),
    'last_txn_date': [(REFERENCE_DATE - pd.Timedelta(days=int(x))).strftime('%Y-%m-%d')
                      for x in rng.integers(1, 180, n)],
    'phone': [f'9{rng.integers(0, 99999999):08d}' for _ in range(n)],
    'email': [f'cliente{i}@banco.test' for i in range(1, n+1)]
})

# --- Introducir errores ---
# Nulos
idx_null_email = rng.choice(n, 15, replace=False)
idx_null_phone = rng.choice(np.setdiff1d(range(n), idx_null_email), 12, replace=False)
idx_null_balance = rng.choice(n, 10, replace=False)
df_raw.loc[idx_null_email, 'email'] = np.nan
df_raw.loc[idx_null_phone, 'phone'] = np.nan
df_raw.loc[idx_null_balance, 'balance'] = np.nan

# Formatos invalidos
idx_bad_email = rng.choice(np.setdiff1d(range(n), idx_null_email), 14, replace=False)
df_raw.loc[idx_bad_email, 'email'] = 'correo_sin_arroba.com'
idx_bad_phone = rng.choice(np.setdiff1d(range(n), idx_null_phone), 12, replace=False)
df_raw.loc[idx_bad_phone, 'phone'] = '123'
idx_bad_doc = rng.choice(n, 10, replace=False)
df_raw.loc[idx_bad_doc, 'document_id'] = '12ABC'

# Balance negativo en depositos
dep_idx = df_raw[df_raw['product'].isin(['Savings', 'Checking', 'TermDeposit'])].index
idx_neg = rng.choice(dep_idx, min(12, len(dep_idx)), replace=False)
df_raw.loc[idx_neg, 'balance'] = -rng.uniform(100, 3000, len(idx_neg)).round(2)

# Fechas futuras
idx_future = rng.choice(n, 8, replace=False)
df_raw.loc[idx_future, 'last_txn_date'] = (REFERENCE_DATE + pd.Timedelta(days=30)).strftime('%Y-%m-%d')

# Premium con saldo bajo
prem_idx = df_raw[df_raw['segment'] == 'Premium'].index
idx_incons = rng.choice(prem_idx, min(10, len(prem_idx)), replace=False)
df_raw.loc[idx_incons, 'balance'] = rng.uniform(50, 500, len(idx_incons)).round(2)

# Duplicados
dup_idx = rng.choice(n, 8, replace=False)
df_raw.loc[dup_idx[:4], 'document_id'] = '12345678'
df_raw.loc[dup_idx[4:], 'document_id'] = '87654321'

print(f"Dataset generado: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas")
print(f"Errores introducidos: nulos, formatos, balance negativo, fechas futuras, inconsistencias, duplicados")
df_raw.head()

Dataset generado: 250 registros, 8 columnas
Errores introducidos: nulos, formatos, balance negativo, fechas futuras, inconsistencias, duplicados


,customer_id,document_id,segment,product,balance,last_txn_date,phone,email
0,CUST00001,82120884,Premium,TermDeposit,10866.43,2026-04-11,980243131,cliente1@banco.test
1,CUST00002,98077071,Standard,CreditCard,2183.59,2026-05-21,123,cliente2@banco.test
2,CUST00003,39342944,Premium,Checking,9161.55,2026-05-31,928268926,cliente3@banco.test
3,CUST00004,54877558,Standard,Savings,2299.51,2026-01-02,939516117,cliente4@banco.test
4,CUST00005,57775721,Mass,Savings,2505.59,2025-12-15,929132945,cliente5@banco.test


## 7. Ejecución del pipeline multi-agente sobre las reglas

Cada regla pasa por: Intérprete → Codificador → Validador → (Corrector si falla)

In [9]:
# Reglas de calidad en lenguaje natural
reglas_nl = [
    "El saldo de una cuenta nunca debe ser negativo para productos de deposito (Savings, Checking, TermDeposit).",
    "El correo electronico debe contener un solo simbolo @ y un dominio valido con punto.",
    "El numero de documento debe tener exactamente 8 digitos numericos.",
    "No puede haber dos clientes con el mismo documento de identidad.",
    "El numero de telefono debe tener exactamente 9 digitos.",
    "Los clientes del segmento Premium deben tener un saldo superior a 1500.",
    "La fecha de ultima transaccion no puede ser posterior a 2026-06-01.",
    "Los campos email y phone no pueden estar vacios."
]

print(f"Procesando {len(reglas_nl)} reglas con pipeline multi-agente...")
print("=" * 70)

results = []
total_start = time.time()

for i, regla in enumerate(reglas_nl, 1):
    print(f"\n--- REGLA {i}/{len(reglas_nl)} ---")
    print(f"    '{regla[:60]}...'")
    
    result = process_rule(
        rule_text=regla,
        func_name=f"clean_rule_{i}",
        df=df_raw,
        max_attempts=4
    )
    results.append(result)

total_time = time.time() - total_start
print(f"\n{'='*70}")
print(f"PIPELINE COMPLETADO en {total_time:.0f}s")
print(f"{'='*70}")

# Resumen
success_count = sum(1 for r in results if r.success)
print(f"Exito: {success_count}/{len(results)} reglas ({100*success_count/len(results):.0f}%)")
print(f"Intentos promedio: {np.mean([r.attempts for r in results]):.1f}")

for r in results:
    status = "OK" if r.success else "FAIL"
    print(f"  [{status}] {r.dimension:15s} | {r.func_name} | intentos: {r.attempts}")
    if not r.success and r.errors_history:
        print(f"        Ultimo error: {r.errors_history[-1][:80]}")

Procesando 8 reglas con pipeline multi-agente...

--- REGLA 1/8 ---
    'El saldo de una cuenta nunca debe ser negativo para producto...'
  [Interprete] Analizando regla...
  [Codificador] Intento 1/4...
  [Validador] Verificando codigo...
  [OK] Codigo valido en intento 1

--- REGLA 2/8 ---
    'El correo electronico debe contener un solo simbolo @ y un d...'
  [Interprete] Analizando regla...
  [Codificador] Intento 1/4...
  [Validador] Verificando codigo...
  [OK] Codigo valido en intento 1

--- REGLA 3/8 ---
    'El numero de documento debe tener exactamente 8 digitos nume...'
  [Interprete] Analizando regla...
  [Codificador] Intento 1/4...
  [Validador] Verificando codigo...
  [OK] Codigo valido en intento 1

--- REGLA 4/8 ---
    'No puede haber dos clientes con el mismo documento de identi...'
  [Interprete] Analizando regla...
  [Codificador] Intento 1/4...
  [Validador] Verificando codigo...
  [OK] Codigo valido en intento 1

--- REGLA 5/8 ---
    'El numero de telefono debe 

## 8. Aplicar limpieza y comparar calidad antes/después

In [10]:
# Aplicar las funciones exitosas al dataset
df_clean = df_raw.copy()
applied_rules = []

for r in results:
    if r.success:
        try:
            safe_ns = {"pd": pd, "np": np, "re": re}
            exec(r.code, safe_ns)
            func = safe_ns[r.func_name]
            df_clean = func(df_clean)
            applied_rules.append(r.func_name)
            print(f"  Aplicada: {r.func_name} ({r.dimension})")
        except Exception as e:
            print(f"  Error aplicando {r.func_name}: {e}")

print(f"\nReglas aplicadas: {len(applied_rules)}/{len(results)}")
print(f"Dataset limpio: {df_clean.shape[0]} registros")

  Aplicada: clean_rule_1 (Validez)
  Aplicada: clean_rule_2 (Validez)
  Aplicada: clean_rule_3 (Validez)
  Aplicada: clean_rule_4 (Unicidad)
  Aplicada: clean_rule_5 (Validez)
  Aplicada: clean_rule_6 (Consistencia)
  Aplicada: clean_rule_7 (Exactitud)

Reglas aplicadas: 7/8
Dataset limpio: 236 registros


In [11]:
# Metricas de calidad antes y despues
def measure_quality(df):
    """Mide calidad por dimension."""
    metrics = {}
    
    # Completitud
    null_pct = df[['email','phone','balance','document_id']].isnull().mean().mean() * 100
    metrics['Completitud'] = round(100 - null_pct, 2)
    
    # Exactitud (balance >= 0 para depositos)
    dep = df[df['product'].isin(['Savings','Checking','TermDeposit'])]
    bal = pd.to_numeric(dep['balance'], errors='coerce')
    if len(dep) > 0:
        metrics['Exactitud'] = round((bal.fillna(0) >= 0).mean() * 100, 2)
    else:
        metrics['Exactitud'] = 100.0
    
    # Consistencia (Premium -> balance > 1500)
    prem = df[df['segment'] == 'Premium']
    if len(prem) > 0:
        bal_p = pd.to_numeric(prem['balance'], errors='coerce')
        metrics['Consistencia'] = round((bal_p.fillna(0) > 1500).mean() * 100, 2)
    else:
        metrics['Consistencia'] = 100.0
    
    # Unicidad
    valid_docs = df['document_id'].dropna()
    if len(valid_docs) > 0:
        metrics['Unicidad'] = round((valid_docs.nunique() / len(valid_docs)) * 100, 2)
    else:
        metrics['Unicidad'] = 100.0
    
    # Validez (email + phone + doc format)
    email_ok = df['email'].fillna('').astype(str).str.fullmatch(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$').mean()
    phone_ok = df['phone'].fillna('').astype(str).str.fullmatch(r'^\d{9}$').mean()
    doc_ok = df['document_id'].fillna('').astype(str).str.fullmatch(r'^\d{8}$').mean()
    metrics['Validez'] = round((email_ok + phone_ok + doc_ok) / 3 * 100, 2)
    
    return pd.Series(metrics)

before = measure_quality(df_raw)
after = measure_quality(df_clean)
comparison = pd.DataFrame({'Antes (%)': before, 'Despues (%)': after,
                           'Mejora (pp)': (after - before).round(2)})

print("\n" + "=" * 60)
print("COMPARACION DE CALIDAD: ANTES vs DESPUES")
print("=" * 60)
print(comparison.to_string())
print(f"\nMejora promedio: {comparison['Mejora (pp)'].mean():.2f} pp")


COMPARACION DE CALIDAD: ANTES vs DESPUES
              Antes (%)  Despues (%)  Mejora (pp)
Completitud       96.40        97.78         1.38
Exactitud         93.56       100.00         6.44
Consistencia      64.71        91.18        26.47
Unicidad          94.40       100.00         5.60
Validez           91.73        98.31         6.58

Mejora promedio: 9.29 pp


In [12]:
# Grafico comparativo
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison['Antes (%)'], width, label='Antes', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, comparison['Despues (%)'], width, label='Despues', color='#27ae60', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=45, ha='right')
ax.set_ylabel('Porcentaje (%)')
ax.set_title('Calidad de datos: antes vs despues (pipeline multi-agente)')
ax.legend()
ax.set_ylim(0, 105)
ax.axhline(95, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('calidad_comparacion.png', dpi=100)
plt.close()
print("Grafico guardado: calidad_comparacion.png")

Grafico guardado: calidad_comparacion.png


## 9. Detección de anomalías (Autoencoder + Isolation Forest)

In [13]:
# Dataset de transacciones
np.random.seed(123)
n_tx = 1200
rng_tx = np.random.default_rng(123)

df_tx = pd.DataFrame({
    'amount': np.round(rng_tx.lognormal(4.5, 0.6, n_tx), 2),
    'hour': rng_tx.integers(0, 24, n_tx),
    'day_of_week': rng_tx.integers(0, 7, n_tx),
    'channel': rng_tx.choice(['ATM','Web','Mobile','Branch'], n_tx, p=[0.25,0.3,0.3,0.15]),
    'country': rng_tx.choice(['PE','PE','PE','CO','US','BR'], n_tx),
})
df_tx['is_anomaly'] = 0

# Insertar anomalias (5%)
anom_idx = rng_tx.choice(n_tx, 60, replace=False)
df_tx.loc[anom_idx[:30], 'amount'] = rng_tx.uniform(50000, 200000, 30).round(2)
df_tx.loc[anom_idx[30:50], 'amount'] = rng_tx.uniform(20000, 80000, 20).round(2)
df_tx.loc[anom_idx[30:50], 'hour'] = rng_tx.choice([1,2,3,4], 20)
df_tx.loc[anom_idx[50:], 'amount'] = rng_tx.uniform(40000, 120000, 10).round(2)
df_tx.loc[anom_idx, 'is_anomaly'] = 1

# Preprocesar
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['amount', 'hour', 'day_of_week']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['channel', 'country'])
])
X = preprocessor.fit_transform(df_tx[['amount','hour','day_of_week','channel','country']])
y_true = df_tx['is_anomaly'].values

# Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X)
y_pred_iso = (iso.predict(X) == -1).astype(int)

# Autoencoder simple con sklearn (MLPRegressor como proxy)
from sklearn.neural_network import MLPRegressor
X_normal = X[y_true == 0]
scaler_ae = MinMaxScaler()
X_norm_scaled = scaler_ae.fit_transform(X_normal)
X_all_scaled = scaler_ae.transform(X)

ae = MLPRegressor(hidden_layer_sizes=(32, 8, 32), max_iter=200,
                  random_state=42, early_stopping=True)
ae.fit(X_norm_scaled, X_norm_scaled)

X_pred = ae.predict(X_all_scaled)
mse = np.mean((X_all_scaled - X_pred) ** 2, axis=1)
threshold = np.percentile(mse[y_true == 0], 95)
y_pred_ae = (mse > threshold).astype(int)

# Metricas
print("DETECCION DE ANOMALIAS")
print("=" * 55)
for name, y_pred in [("Isolation Forest", y_pred_iso), ("Autoencoder", y_pred_ae)]:
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  F1-Score:  {f1_score(y_true, y_pred):.4f}")

DETECCION DE ANOMALIAS

Isolation Forest:
  Accuracy:  0.9767
  Precision: 0.7667
  Recall:    0.7667
  F1-Score:  0.7667

Autoencoder:
  Accuracy:  0.9525
  Precision: 0.5128
  Recall:    1.0000
  F1-Score:  0.6780


## 10. Predicción de errores futuros

In [14]:
# Features para prediccion
np.random.seed(77)
n_pred = 500
rng_p = np.random.default_rng(77)

df_pred = pd.DataFrame({
    'balance': rng_p.exponential(5000, n_pred).round(2),
    'segment_code': rng_p.choice([0,1,2,3], n_pred, p=[0.35,0.35,0.2,0.1]),
    'product_code': rng_p.choice([0,1,2,3], n_pred),
    'phone_length': rng_p.choice([3,5,9,9,9,9,10], n_pred),
    'email_has_at': rng_p.choice([0,1,1,1,1,1], n_pred),
    'doc_length': rng_p.choice([3,5,8,8,8,8,8], n_pred),
    'n_prev_errors': rng_p.poisson(1, n_pred),
    'days_since_update': rng_p.exponential(30, n_pred).astype(int)
})
df_pred['has_issue'] = (
    (df_pred['phone_length'] < 9) | (df_pred['email_has_at'] == 0) |
    (df_pred['doc_length'] != 8) | (df_pred['n_prev_errors'] > 3) |
    ((df_pred['segment_code'] == 2) & (df_pred['balance'] < 1500))
).astype(int)

X_p = df_pred.drop('has_issue', axis=1)
y_p = df_pred['has_issue']
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.3, random_state=42, stratify=y_p)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

print("PREDICCION DE ERRORES FUTUROS")
print("=" * 55)
print(f"Accuracy:  {accuracy_score(y_te, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_te, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_te, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_te, y_pred_rf):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_te, y_proba):.4f}")

PREDICCION DE ERRORES FUTUROS
Accuracy:  0.9667
Precision: 1.0000
Recall:    0.9468
F1-Score:  0.9727
AUC-ROC:   1.0000


## 11. Resumen de efectividad del sistema completo

In [15]:
print("=" * 70)
print("RESUMEN FINAL DE EFECTIVIDAD — SISTEMA MULTI-AGENTE")
print("=" * 70)

# 1. LLM Pipeline
print("\n1. PIPELINE LLM MULTI-AGENTE")
print("-" * 40)
success_rate = 100 * success_count / len(results)
avg_attempts = np.mean([r.attempts for r in results])
print(f"   Tasa de exito: {success_count}/{len(results)} ({success_rate:.0f}%)")
print(f"   Intentos promedio: {avg_attempts:.1f}")
print(f"   Reglas por dimension:")
for dim in set(r.dimension for r in results):
    dim_results = [r for r in results if r.dimension == dim]
    dim_ok = sum(1 for r in dim_results if r.success)
    print(f"     {dim}: {dim_ok}/{len(dim_results)}")

# 2. Calidad
print("\n2. MEJORA EN CALIDAD DE DATOS")
print("-" * 40)
print(comparison.to_string())

# 3. Anomalias
print("\n3. DETECCION DE ANOMALIAS")
print("-" * 40)
print(f"   Isolation Forest: F1={f1_score(y_true, y_pred_iso):.4f}")
print(f"   Autoencoder:      F1={f1_score(y_true, y_pred_ae):.4f}")

# 4. Prediccion
print("\n4. PREDICCION DE ERRORES")
print("-" * 40)
print(f"   Random Forest: F1={f1_score(y_te, y_pred_rf):.4f}, AUC={roc_auc_score(y_te, y_proba):.4f}")

# 5. RAG
print("\n5. BASE DE CONOCIMIENTO RAG")
print("-" * 40)
print(f"   Ejemplos exitosos acumulados: {len(SUCCESSFUL_EXAMPLES)}")
print(f"   Estos ejemplos se inyectan en prompts subsiguientes para mejorar")
print(f"   la calidad de generacion (aprendizaje continuo en sesion).")

print(f"\n{'='*70}")
print("FIN DEL NOTEBOOK v4 — Sistema Multi-Agente con RAG")
print(f"{'='*70}")

RESUMEN FINAL DE EFECTIVIDAD — SISTEMA MULTI-AGENTE

1. PIPELINE LLM MULTI-AGENTE
----------------------------------------
   Tasa de exito: 7/8 (88%)
   Intentos promedio: 1.4
   Reglas por dimension:
     Consistencia: 1/1
     Unicidad: 1/1
     Completitud: 0/1
     Validez: 4/4
     Exactitud: 1/1

2. MEJORA EN CALIDAD DE DATOS
----------------------------------------
              Antes (%)  Despues (%)  Mejora (pp)
Completitud       96.40        97.78         1.38
Exactitud         93.56       100.00         6.44
Consistencia      64.71        91.18        26.47
Unicidad          94.40       100.00         5.60
Validez           91.73        98.31         6.58

3. DETECCION DE ANOMALIAS
----------------------------------------
   Isolation Forest: F1=0.7667
   Autoencoder:      F1=0.6780

4. PREDICCION DE ERRORES
----------------------------------------
   Random Forest: F1=0.9727, AUC=1.0000

5. BASE DE CONOCIMIENTO RAG
----------------------------------------
   Ejemplos exito

## 12. Conclusiones y trabajo futuro

**Arquitectura demostrada:**
- Pipeline multi-agente (4 agentes especializados) reduce errores del LLM
- Validación AST previene ejecución de código inseguro
- RAG con ejemplos exitosos mejora la generación iterativamente
- Fine-tuning vía Modelfile adapta el LLM al dominio bancario

**Limitaciones:**
- Datos sintéticos (no representan complejidad real)
- Fine-tuning limitado a Modelfile (no QLoRA real por hardware)
- Latencia alta en CPU (~30-80s por regla)

**Trabajo futuro:**
- QLoRA completo en GPU (Google Colab o cloud)
- RAG persistente entre sesiones (vectorstore)
- Más reglas de negocio del banco real
- Integración con la aplicación web